# PARC2026 — π0.5 smoke + GA=8 verification

このNotebookは、運営GPUで本学習を始める前の **Gate O1** をColab A100で潰すためのものです。

目的は2つです。
1. π0.5 LoRAの短い学習をColabで再現する。
2. `GA=8` が **8 micro-step → 1 optimizer update** になっていることをruntime traceで確認する。

HF tokenは画面入力し、Notebookへ保存しません。最初は3 optimizer stepだけでGAを検証し、20-step + mergeは明示的に有効化した場合だけ実行します。

In [ ]:
from pathlib import Path
import os, platform, shutil, subprocess, sys

ROOT = Path('/content/parc2026')
REPO = ROOT / 'py_AI'
PI05_DIR = REPO / 'examples/pi05_libero_finetune'
assert REPO.exists(), '00_a100_preflight.ipynb を先に実行してください'
print('repo:', REPO)
print('git:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip())
print('runtime python:', sys.version)
print('platform:', platform.platform())
subprocess.run(['nvidia-smi'], check=True)

## Python 3.10を用意する
採点側Python 3.10とLeRobot v0.4.4の組み合わせへ寄せるため、Colab runtimeのPython版に依存せず `uv` でPython 3.10を用意します。

In [ ]:
if shutil.which('uv') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'python', 'install', '3.10'], check=True)
PY310 = subprocess.check_output(['uv', 'python', 'find', '3.10'], text=True).strip()
print('python3.10:', PY310)
subprocess.run([PY310, '--version'], check=True)

## DatasetとHF token
`DATASET_ROOT` は **LeRobot形式のdataset root (`meta/`, `data/`, `videos/`)** を指定します。運営combined datasetをColabへまだ置いていない場合、このセルで止まるのが正常です。

tokenは `getpass` で入力し、出力にもNotebookにも残しません。

In [ ]:
from getpass import getpass

DATASET_ROOT = Path(os.environ.get('PI05_DATASET_ROOT', '/content/parc2026/datasets/libero_combined_20hz'))
DATASET_REPO_ID = os.environ.get('PI05_DATASET_REPO_ID', 'local/libero_combined_20hz')
print('dataset root:', DATASET_ROOT)
if not (DATASET_ROOT / 'meta').exists():
    raise FileNotFoundError(
        f'{DATASET_ROOT} に meta/ がありません。'
        ' 運営datasetまたは検証用LeRobot datasetを配置して PI05_DATASET_ROOT を設定してください。'
    )
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass('HF token (PaliGemma access権があるtoken): ')
print('HF_TOKEN: set (not displayed)')

## 学習環境setup
venv / HF cache / outputsはColabの `/content` 側へ置き、Google Driveやrepoへ巨大cacheを置きません。

In [ ]:
TRAIN_DATA_ROOT = ROOT / 'cache' / 'pi05-train'
LEROBOT_ROOT = ROOT / 'vendor' / 'lerobot-pi05'
TRAIN_DATA_ROOT.mkdir(parents=True, exist_ok=True)
env = os.environ.copy()
env.update({
    'PYTHON': PY310,
    'DATA_ROOT': str(TRAIN_DATA_ROOT),
    'LEROBOT_ROOT': str(LEROBOT_ROOT),
})
subprocess.run(['bash', 'scripts/setup_train.sh'], cwd=PI05_DIR, env=env, check=True)

## GA=8 runtime probe
3 optimizer stepだけ回します。`sitecustomize.py` を `PYTHONPATH` へ注入し、backward、Accelerate optimizer call、実optimizer step、scheduler stepをJSONLへ記録します。

期待値: **3 optimizer step × GA8 = 24 micro-step**。

In [ ]:
GA = 8
PROBE_STEPS = 3
PROBE_BS = 1
TRACE = ROOT / 'outputs' / 'pi05_ga8_probe_trace.jsonl'
TRACE_SUMMARY = ROOT / 'outputs' / 'pi05_ga8_probe_summary.json'
TRACE.parent.mkdir(parents=True, exist_ok=True)
TRACE.unlink(missing_ok=True)
TRACE_SUMMARY.unlink(missing_ok=True)

probe_env = os.environ.copy()
probe_env.update({
    'PARC_GA_TRACE_FILE': str(TRACE),
    'PYTHONPATH': str(REPO / 'tools/pi05/ga_instrument') + (':' + probe_env['PYTHONPATH'] if probe_env.get('PYTHONPATH') else ''),
    'PI05_DATASET_ROOT': str(DATASET_ROOT),
    'PI05_DATASET_REPO_ID': DATASET_REPO_ID,
    'PI05_VIDEO_BACKEND': os.environ.get('PI05_VIDEO_BACKEND', 'pyav'),
    'SMOKE_BS': str(PROBE_BS),
    'SMOKE_GA': str(GA),
    'SMOKE_STEPS': str(PROBE_STEPS),
    'SMOKE_SKIP_MERGE': '1',
    'RUN_NAME': 'colab_ga8_probe',
})
cmd = 'source env_train.sh && bash scripts/smoke_pi05.sh'
subprocess.run(['bash', '-lc', cmd], cwd=PI05_DIR, env=probe_env, check=True)
print('trace:', TRACE)

In [ ]:
summary_cmd = [
    sys.executable, str(REPO / 'tools/pi05/summarize_ga_trace.py'), str(TRACE),
    '--expected-ga', str(GA), '--expected-steps', str(PROBE_STEPS),
    '--json-out', str(TRACE_SUMMARY),
]
subprocess.run(summary_cmd, check=True)
print('GA Gate: PASS')
print('summary:', TRACE_SUMMARY)

## Optional: 20-step smoke + LoRA merge
GA probeがPASSした後だけ実施します。既定は `False` なので、必要になった時に `True` へ変更してください。

In [ ]:
RUN_FULL_20_STEP = False

if RUN_FULL_20_STEP:
    full_env = os.environ.copy()
    full_env.update({
        'PI05_DATASET_ROOT': str(DATASET_ROOT),
        'PI05_DATASET_REPO_ID': DATASET_REPO_ID,
        'PI05_VIDEO_BACKEND': os.environ.get('PI05_VIDEO_BACKEND', 'pyav'),
        'SMOKE_BS': '2',
        'SMOKE_GA': '8',
        'SMOKE_STEPS': '20',
        'RUN_NAME': 'colab_pi05_smoke_ga8_20',
    })
    subprocess.run(
        ['bash', '-lc', 'source env_train.sh && bash scripts/smoke_pi05.sh'],
        cwd=PI05_DIR, env=full_env, check=True,
    )
    print('20-step smoke + merge: PASS')
else:
    print('skip: RUN_FULL_20_STEP=False')

## Exit criteria
このNotebookの必須Gateは `GA Gate: PASS` です。

PASS後に保存するもの: `pi05_ga8_probe_summary.json` と、必要なら20-step smokeのmerged checkpoint。
次は同じdataset/eval contractへSmolVLA/OpenVLA-OFTを接続します。